<div style="display: flex; align-items: center; gap: 18px; margin-bottom: 15px;">
  <img src="https://files.codebasics.io/v3/images/sticky-logo.svg" alt="Codebasics Logo" style="display: inline-block;" width="130">
  <h1 style="font-size: 34px; color: #1f4e79; margin: 0; display: inline-block;">Codebasics Practice Room - Data Engineering Bootcamp </h1>
</div>

## 🧑🏼‍🔧 Setup

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
from delta.tables import DeltaTable

In [0]:
# --------------------------------------------
# ⚙️ Databricks Unity Catalog Setup (Auto)
# --------------------------------------------
from pyspark.sql import SparkSession

catalog_name = "practice_db_catalog"
schema_name = "airbnb"
volume_name = "data_volume"

# 1️⃣ Create Catalog if not exists
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
print(f"✅ Catalog `{catalog_name}` ready.")

# 2️⃣ Create Schema (Database) if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
print(f"✅ Schema `{schema_name}` created inside `{catalog_name}`.")

# 3️⃣ Create Volume if not exists
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
print(f"✅ Volume `{volume_name}` created inside `{catalog_name}.{schema_name}`")

# 4️⃣ Set current context
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {schema_name}")

# 5️⃣ Define volume-backed paths
base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/airbnb"
raw_path = f"{base_path}/raw"
clean_path = f"{base_path}/clean"
silver_path = f"{base_path}/silver"

# 6️⃣ Create directories inside the volume
dbutils.fs.mkdirs(raw_path)
dbutils.fs.mkdirs(clean_path)
dbutils.fs.mkdirs(silver_path)

print("✅ Paths initialized successfully:")
print(f"Raw: {raw_path}")
print(f"Clean: {clean_path}")
print(f"Silver: {silver_path}")


✅ Catalog `practice_db_catalog` ready.
✅ Schema `airbnb` created inside `practice_db_catalog`.
✅ Volume `data_volume` created inside `practice_db_catalog.airbnb`
✅ Paths initialized successfully:
Raw: /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw
Clean: /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/clean
Silver: /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver


In [0]:
# 🧮 Generate Airbnb listings dataset (Spark-native version)
from pyspark.sql import Row
import random, datetime

# --------------------------------
# Configuration
# --------------------------------

random.seed(42) # ✅ reproducibility

num_records = 600  # Adjust as needed

amenities_pool = [
    "Wifi", "Kitchen", "Washer", "Dryer", "TV", "Essentials", "Air conditioning",
    "Heating", "Pool", "Hot tub", "Balcony", "Garden", "Parking", "Fireplace",
    "Sea view", "Mountain view", "Pet friendly", "Gym", "Breakfast", "Workspace"
]

property_types = [
    "Studio Apartment", "Private Room", "Entire Home", "Cottage", "Villa",
    "Cabin", "Loft", "Guest Suite", "Bungalow", "Condo"
]

cities = ["Mumbai", "Bangalore", "Hyderabad", "Chennai", "Pune", "Delhi", "Goa"]
boolean_variants = [True, False, "true", "false", "Yes", "No", "yes", "no", "TRUE", "FALSE"]

# --------------------------------
# Generate data as list of Rows
# --------------------------------
data = []
for i in range(1, num_records + 1):
    created_date = datetime.date(2025, 1, 1) + datetime.timedelta(days=random.randint(0, 300))
    last_booked_date = created_date + datetime.timedelta(days=random.randint(1, 60))

    data.append(Row(
        id=100 + i,
        name=f"{random.choice(['Cozy', 'Modern', 'Luxury', 'Spacious', 'Budget'])} "
             f"{random.choice(property_types)} in {random.choice(cities)}",
        city=random.choice(cities),
        price_per_night=random.randint(1000, 10000),
        amenities=random.sample(amenities_pool, random.randint(3, 8)),
        has_parking=random.choice(boolean_variants),
        is_superhost=random.choice(boolean_variants),
        created_date=str(created_date),
        last_booked_date=str(last_booked_date)
    ))

# --------------------------------
# Convert to Spark DataFrame
# --------------------------------
df_raw = spark.createDataFrame(data)

# --------------------------------
# Write directly to UC Volume (JSON format)
# --------------------------------
raw_path = "/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json"

df_raw.write.mode("overwrite").json(raw_path)
print(f"✅ Successfully generated {num_records} Airbnb listings and saved to:")
print(f"📂 {raw_path}")

# --------------------------------
# Quick sanity check
# --------------------------------


✅ Successfully generated 600 Airbnb listings and saved to:
📂 /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json



# ❓ Scenario Question: Airbnb — Clean Listing Amenities (PySpark) [Easy]



## 🗂️ Scenario

You are working with raw **Airbnb listing data** ingested from multiple sources.  
Each listing contains property details and a **nested list of amenities**.  
The goal is to **clean, normalize, and store** this data for downstream analysis.

The data is available as a JSON file (`listings.json`) in the **Bronze layer**, which now needs to be transformed into a clean **Silver Delta Table**.

---

## 🎯 Task

Perform the following transformations:

1. **Read** the input data from `listings.json` using Spark.  
2. **Explode** the `amenities` array so that each row contains a single amenity.  
3. **Normalize** boolean-like columns (e.g., `"true"`, `"false"`, `"yes"`, `"no"`) into proper boolean (`True` / `False`) Spark data types.  
4. **Rename** or select only the relevant columns for downstream use.  
5. **Save** the cleaned DataFrame in **Delta format** to the **Silver layer** path:  
   `/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/listings.json` 

---

## 🧩 Assumptions

- The input file `listings.json` exists in the **Bronze** path:  
  `/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json`
- The `amenities` field may contain an array or a stringified array.  
- Boolean columns may contain values like `"TRUE"`, `"Yes"`, `"0"`, `"1"`, etc.  
- The final cleaned DataFrame should contain only essential columns:  
  `id`, `name`, `amenity`, `has_parking`, and `is_superhost`.  
- Handle missing or malformed columns gracefully (e.g., cast to `null`).  

---

## 📦 Deliverables

- **Output Format:** Delta table written to Silver  
- **Output Path:** `/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/listings.json`

| **Expected Columns** | `id`, `name`, `amenity`, `has_parking`, `is_superhost` |

---

## 🧠 Notes

- Use `pyspark.sql.functions.explode()` to expand the amenities array.  
- Use `F.when()` or `F.col().cast("boolean")` for boolean normalization.  
- Use clear column aliases for readability.  
- Validate the write by reading from the Silver path and displaying the first few rows.

---

## 🧩 Example Output (simplified)

| id  | name               | amenity          | has_parking | is_superhost |
|-----|--------------------|------------------|--------------|---------------|
| 101 | Cozy Beach House   | Wifi             | true         | false         |
| 101 | Cozy Beach House   | Ocean View       | true         | false         |
| 102 | City Apartment     | Air Conditioning | false        | true          |


## 🛢️Input data

In [0]:
display(df_raw.limit(5))

id,name,city,price_per_night,amenities,has_parking,is_superhost,created_date,last_booked_date
101,Luxury Cottage in Bangalore,Bangalore,2679,"List(Gym, Washer, Fireplace, Kitchen, Wifi, Pet friendly, Dryer, Workspace)",TRUE,FALSE,2025-02-27,2025-03-01
102,Modern Bungalow in Chennai,Bangalore,8359,"List(Pool, Wifi, Essentials, Fireplace, Balcony, TV, Washer)",false,No,2025-01-14,2025-02-19
103,Spacious Private Room in Hyderabad,Goa,6635,"List(Pool, Kitchen, Sea view, Dryer, Parking, Breakfast, Workspace)",Yes,FALSE,2025-02-22,2025-02-28
104,Modern Private Room in Mumbai,Delhi,4733,"List(Washer, Heating, Dryer, Parking, Pool)",no,No,2025-07-05,2025-08-11
105,Luxury Cottage in Delhi,Hyderabad,2169,"List(Essentials, Gym, Heating, Workspace, Sea view, Air conditioning, TV)",TRUE,false,2025-03-25,2025-04-18


# 📝 Your Solution

In [0]:
# ✍️ Your Solution Here

from pyspark.sql import functions as F

# Steps:
# 1. Read the JSON file

df= spark.read.json("/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json")
display(data)
# 2. Explode the amenities


# 3. Normalize boolean-like fields and retrun the dataframe


amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965
"List(Dryer, Pet friendly, Washer)",Delhi,2025-03-20,true,478,true,2025-04-29,Luxury Cabin in Chennai,6276
"List(Kitchen, Breakfast, Parking)",Chennai,2025-04-02,no,479,TRUE,2025-05-28,Cozy Cottage in Mumbai,4139
"List(Air conditioning, Mountain view, Washer)",Delhi,2025-05-12,no,480,true,2025-05-16,Luxury Cottage in Pune,2161
"List(Pool, Pet friendly, Kitchen, Washer, Hot tub, Gym, Dryer)",Delhi,2025-07-04,TRUE,481,false,2025-08-09,Spacious Cabin in Goa,1955
"List(Heating, Breakfast, Essentials, Fireplace, Dryer, Parking, Balcony)",Bangalore,2025-02-21,FALSE,482,No,2025-04-09,Modern Bungalow in Chennai,4164
"List(Parking, Pet friendly, Balcony, Essentials, Sea view, Mountain view, Breakfast)",Goa,2025-08-23,No,483,TRUE,2025-10-07,Spacious Private Room in Bangalore,3955
"List(Balcony, Heating, Gym, Air conditioning, Parking, Mountain view, Pool, Garden)",Delhi,2025-06-16,Yes,484,No,2025-08-10,Luxury Loft in Goa,8054
"List(Wifi, Pet friendly, TV, Hot tub, Workspace, Gym, Dryer)",Mumbai,2025-08-20,false,485,No,2025-09-18,Spacious Studio Apartment in Delhi,5660


## 🔍 Validation Questions

After creating the final DataFrame (`df_final`), answer these to check your understanding:

1. How many amenities are listed for the property with **ID = 101**?  
2. How many listings have **`is_superhost = true`**?  
3. What are the **unique amenities** available for listing **ID = 103**?  
4. Count how many listings have **`has_parking = true`**.  
5. For each listing, how many total amenities are available? (Hint: use `groupBy().count()`.)

In [0]:
# Steps:
# 1. Read the JSON file

df= spark.read.json("/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/raw/listings.json")
display(data)

amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965
"List(Dryer, Pet friendly, Washer)",Delhi,2025-03-20,true,478,true,2025-04-29,Luxury Cabin in Chennai,6276
"List(Kitchen, Breakfast, Parking)",Chennai,2025-04-02,no,479,TRUE,2025-05-28,Cozy Cottage in Mumbai,4139
"List(Air conditioning, Mountain view, Washer)",Delhi,2025-05-12,no,480,true,2025-05-16,Luxury Cottage in Pune,2161
"List(Pool, Pet friendly, Kitchen, Washer, Hot tub, Gym, Dryer)",Delhi,2025-07-04,TRUE,481,false,2025-08-09,Spacious Cabin in Goa,1955
"List(Heating, Breakfast, Essentials, Fireplace, Dryer, Parking, Balcony)",Bangalore,2025-02-21,FALSE,482,No,2025-04-09,Modern Bungalow in Chennai,4164
"List(Parking, Pet friendly, Balcony, Essentials, Sea view, Mountain view, Breakfast)",Goa,2025-08-23,No,483,TRUE,2025-10-07,Spacious Private Room in Bangalore,3955
"List(Balcony, Heating, Gym, Air conditioning, Parking, Mountain view, Pool, Garden)",Delhi,2025-06-16,Yes,484,No,2025-08-10,Luxury Loft in Goa,8054
"List(Wifi, Pet friendly, TV, Hot tub, Workspace, Gym, Dryer)",Mumbai,2025-08-20,false,485,No,2025-09-18,Spacious Studio Apartment in Delhi,5660


In [0]:
#2. Explode the amenities?
from pyspark.sql.functions import explode
df_explode=df.withColumn("amenity",explode("amenities"))
display(df_explode)


amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night,amenity
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Sea view
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Hot tub
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Pool
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,no,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Garden
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Heating
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Balcony
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Washer
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Parking
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Air conditioning
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,no,477,TRUE,2025-10-12,Modern Studio Apartment in Pune,6965,Pool


In [0]:
# Normalize boolean-like fields and retrun the dataframe#
df.describe()

DataFrame[summary: string, city: string, created_date: string, has_parking: string, id: string, is_superhost: string, last_booked_date: string, name: string, price_per_night: string]

In [0]:
df.printSchema()

root
 |-- amenities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- city: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- has_parking: string (nullable = true)
 |-- id: long (nullable = true)
 |-- is_superhost: string (nullable = true)
 |-- last_booked_date: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price_per_night: long (nullable = true)



In [0]:
df.select("has_parking","is_superhost").distinct().show()


+-----------+------------+
|has_parking|is_superhost|
+-----------+------------+
|       TRUE|        TRUE|
|      false|          no|
|       TRUE|       false|
|       true|          No|
|       TRUE|       FALSE|
|         no|          no|
|         no|          No|
|        Yes|       FALSE|
|         no|        true|
|      FALSE|          No|
|         no|       false|
|         No|          no|
|         No|        TRUE|
|      FALSE|       FALSE|
|         No|         yes|
|         No|       false|
|        Yes|         yes|
|       true|         yes|
|      FALSE|       false|
|         no|        TRUE|
+-----------+------------+
only showing top 20 rows


In [0]:
#Normalize boolean-like columns (e.g., "true", "false", "yes", "no") into proper boolean (True / False) Spark data types.
from pyspark.sql.functions import col, when, lower, trim

boolean_columns = [
    "has_parking",
    "is_superhost"
]

df_clean = df_explode

for c in boolean_columns:
    df_clean = df_clean.withColumn(
        c,
        when(lower(trim(col(c))).isin("true", "yes"), True)
        .when(lower(trim(col(c))).isin("false", "no"), False)
        .otherwise(None))
display(df_clean.limit(5))  


amenities,city,created_date,has_parking,id,is_superhost,last_booked_date,name,price_per_night,amenity
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Sea view
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Hot tub
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Pool
"List(Sea view, Hot tub, Pool, Garden)",Mumbai,2025-05-21,false,476,true,2025-07-16,Budget Bungalow in Bangalore,9702,Garden
"List(Heating, Balcony, Washer, Parking, Air conditioning, Pool, Breakfast)",Goa,2025-09-17,false,477,true,2025-10-12,Modern Studio Apartment in Pune,6965,Heating


In [0]:
df.printSchema()

root
 |-- amenities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- city: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- has_parking: string (nullable = true)
 |-- id: long (nullable = true)
 |-- is_superhost: string (nullable = true)
 |-- last_booked_date: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price_per_night: long (nullable = true)



In [0]:
# Step 4: Select required columns
# The final cleaned DataFrame should contain only essential columns:
# id, name, amenity, has_parking, and is_superhost.
df_final = df_clean.select(
    col("id"),
    col("name"),
    col("has_parking"),
    col("is_superhost"),
    col("amenity")
)

display(df_final)




In [0]:
# Save the cleaned DataFrame in Delta format to the Silver layer path:
# /Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/listings.json
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/practice_db_catalog/airbnb/data_volume/airbnb/silver/listings")


In [0]:
display(df_final)

id,name,has_parking,is_superhost,amenity
476,Budget Bungalow in Bangalore,false,true,Sea view
476,Budget Bungalow in Bangalore,false,true,Hot tub
476,Budget Bungalow in Bangalore,false,true,Pool
476,Budget Bungalow in Bangalore,false,true,Garden
477,Modern Studio Apartment in Pune,false,true,Heating
477,Modern Studio Apartment in Pune,false,true,Balcony
477,Modern Studio Apartment in Pune,false,true,Washer
477,Modern Studio Apartment in Pune,false,true,Parking
477,Modern Studio Apartment in Pune,false,true,Air conditioning
477,Modern Studio Apartment in Pune,false,true,Pool


In [0]:
#  Validation Questions
# After creating the final DataFrame (df_final), answer these to check your understanding:

# How many amenities are listed for the property with ID = 101?
# How many listings have is_superhost = true?
# What are the unique amenities available for listing ID = 103?
# Count how many listings have has_parking = true.
# For each listing, how many total amenities are available? (Hint: use groupBy().count().)

In [0]:
df_final.filter(col("id") == 101).count()

8

In [0]:
from pyspark.sql.functions import col

df_final.filter(col("is_superhost") == True) \
    .select("id") \
    .distinct() \
    .count()

316

In [0]:
#How many listings have is_superhost = true? (e.g., 28)
df_final.filter(col("is_superhost") == True).count()


1753

In [0]:
df_final.filter(col("is_superhost") == True).count()



1753

In [0]:
df_final.filter(col("id") == 103).select("amenity").distinct().count()

7

In [0]:
df_final.filter(col("has_parking") == True).count()

1618

In [0]:
# For each listing, how many total amenities are available? (Hint: use groupBy().count().)
df_final.groupBy("id").count().show()

+---+-----+
| id|count|
+---+-----+
|480|    3|
|501|    4|
|517|    4|
|519|    7|
|490|    6|
|498|    7|
|510|    6|
|514|    7|
|530|    4|
|542|    4|
|489|    8|
|497|    8|
|520|    6|
|544|    6|
|483|    7|
|506|    7|
|507|    7|
|531|    8|
|478|    3|
|482|    7|
+---+-----+
only showing top 20 rows
